# Model Eval; Grade Visualisation

## How to run the grading

Grading is performed by Claude Code using `data/results/grading_spec.md` as the scoring guide.

**Prerequisite:** `data/results/model_eval.json` must exist; produced by running `04_model_eval.ipynb`.

**Run the grading** from the project root:

```
claude
```

Then paste this prompt:

```
Read `data/results/grading_spec.md` and `data/results/model_eval.json`, then grade every
record according to the criteria below. Write the results to `data/results/model_eval_grades.json`.
```

Claude reads both files, scores each record on the task-specific criteria (1–5), and writes
`data/results/model_eval_grades.json`; metadata + scores only, no raw results.

Once that file is present, run the cells below.

In [7]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

with open('../data/results/model_eval_grades.json') as f:
    records = json.load(f)

rows = []
for r in records:
    base = {
        'model': r['model'],
        'task': r['task'],
        'doc': r['doc'],
        'elapsed_s': r['elapsed_s'],
    }
    for k, v in r['scores'].items():
        if k != 'notes':
            rows.append({**base, 'criterion': k, 'score': v})

df = pd.DataFrame(rows)
print(df.shape, df['model'].nunique(), 'models')

(616, 6) 28 models


In [11]:
# Heatmap: average score per model × criterion
pivot = df.groupby(['model', 'criterion'])['score'].mean().unstack()

# Order criteria by task group for readability
criterion_order = [
    'entity_form', 'type_accuracy', 'hallucination',   # NER
    'relevance', 'style', 'language', 'coverage',      # tags (coverage shared with summary)
    'factual_accuracy', 'specificity',                  # summary
]
criterion_order = [c for c in criterion_order if c in pivot.columns]
pivot = pivot[criterion_order]

# Order models alphabetically
model_order = sorted(pivot.index.tolist())
pivot = pivot.loc[model_order]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmin=1, zmax=5,
    text=pivot.round(1).values,
    texttemplate='%{text}',
    hovertemplate='model: %{y}criterion: %{x}avg score: %{z:.2f}',
))
fig.update_layout(
    title='Average score per model × criterion (across both docs)',
    xaxis_title='Criterion',
    yaxis_title='Model',
    height=800,
    margin=dict(l=140),
)
fig.show()

In [12]:
# Grouped bar chart: mean score per model, faceted by task
task_mean = (
    df.groupby(['model', 'task'])['score']
    .mean()
    .reset_index()
    .rename(columns={'score': 'mean_score'})
)

# Order models alphabetically
model_order = sorted(df['model'].unique().tolist())

fig = px.bar(
    task_mean,
    x='model',
    y='mean_score',
    facet_col='task',
    category_orders={'model': model_order, 'task': ['ner', 'tags', 'summary']},
    color='mean_score',
    color_continuous_scale='RdYlGn',
    range_color=[1, 5],
    labels={'mean_score': 'Mean score', 'model': 'Model'},
    title='Mean score per task and model (averaged across both docs and task criteria)',
    height=400,
)
fig.update_layout(xaxis_tickangle=-40, coloraxis_showscale=False)
fig.update_yaxes(range=[0, 5.2])
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1].upper()))
fig.show()

In [13]:
# Scatter: mean elapsed_s vs mean quality score (speed / quality tradeoff)
agg = (
    df.groupby('model')
    .agg(mean_score=('score', 'mean'), mean_elapsed=('elapsed_s', 'mean'))
    .reset_index()
)

fig = px.scatter(
    agg,
    x='mean_elapsed',
    y='mean_score',
    text='model',
    labels={'mean_elapsed': 'Mean elapsed (s)', 'mean_score': 'Mean score'},
    title='Speed vs quality tradeoff per model',
    color='mean_score',
    color_continuous_scale='RdYlGn',
    range_color=[1, 5],
    height=450,
)
fig.update_traces(textposition='top center', marker_size=10)
fig.update_layout(yaxis_range=[1, 5.2], coloraxis_showscale=False)
fig.show()

## Conclusion

Grades cover 28 models across 3 tasks (NER, tags, summary) and 2 documents (EN + NL), yielding 168 records and 616 individual criterion scores.

### Best model: `gemma3:12b` — overall mean 4.68 / 5

Strongest performer across all three tasks: NER 4.00, tags 4.88, summary 5.00. Tags were clean short atomic terms with excellent coverage; summaries were factually accurate, comprehensive, and specific on every record. NER entity forms were generally clean though honorifics appeared. Mean latency was 14.7 s — the fastest among the top tier.

### Second best: `phi4:14b` — overall mean 4.50 / 5

Matched `gemma3:12b` on summary quality (5.00) and NER (4.00), but scored lower on tags (4.38) due to verbose compound terms like `nationally-determined-contributions` and sparse tag sets for the NL document. Latency was roughly double at 31.1 s.

### Notable observations

- **Failed runs**: `olmo-3:7b`, `qwen3-vl:30b`, `falcon:7b`, and several `qwen3` / `deepseek-r1:8b` records returned null or entirely wrong output, scoring at the bottom of the ranking.
- **Hallucinations**: Only `rnj-1:8b` and `falcon:7b` invented entities not present in the source. All other models were hallucination-free on NER.
- **NER entity form** was the weakest criterion across the board: most models included honorifics (Dr., Professor) or appended context phrases ("Danish wind energy company Ørsted"), pulling scores down to 3/5 on average.
- **Tags language**: `olmo2:13b` was the only model to output non-English tags for the NL document; all others correctly used English regardless of source language.
- **Summary factual errors**: `mistral:7b` misidentified Brazil as a top-3 emitter; `granite3.3:8b` called India "second-largest" (source says third); `granite4:7b-a1b-h` NL confused the 1.5°C and 2°C thresholds.